# Arquitecturas Multi-Agente con LangGraph

En el notebook anterior construiste el agentic loop. Ahora lo usarás como bloque
de construcción para sistemas más complejos.

## El problema del agente monolítico

Un solo agente con demasiadas responsabilidades se vuelve:
- **Difícil de depurar** — ¿en qué paso falló?
- **Poco confiable** — un prompt largo confunde al LLM
- **No reutilizable** — no puedes extraer solo la parte de "investigación"

## Patrones implementados

| Patrón | Cuándo usarlo | Implementación LangGraph |
|---|---|---|
| **Orquestador LLM** | El orden es dinámico | Sub-agentes como `@tool` |
| **Sequential** | Pipeline fijo, orden garantizado | Nodos en cadena |
| **Parallel** | Tareas independientes | `Send()` fan-out |
| **Loop** | Refinamiento iterativo | Ciclo + edge condicional |
| **Evaluación** | Medir calidad del sistema | Métricas de proceso + LLM-as-judge |

---
## ⚙️ Sección 1: Instalación

In [ ]:
# pip install langgraph langchain-openai langchain-core httpx
%pip install langgraph langchain-openai langchain-core httpx --quiet

---
## 🔧 Sección 2: Setup

Misma celda de configuración multi-backend que en Day 1A.
Cambia `BACKEND` para elegir tu proveedor.

In [6]:
import os
import time
import httpx
import urllib.parse
from typing import Literal, Annotated, TypedDict
import operator
from dotenv import load_dotenv
from ddgs import DDGS

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, BaseMessage
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState, add_messages
from langgraph.prebuilt import ToolNode


# ════════════════════════════════════════════════════════════
#  CONFIGURACIÓN — cambia BACKEND para elegir tu proveedor
# ════════════════════════════════════════════════════════════

load_dotenv()

BACKEND = "nvidia"   # opciones: "ollama" | "anthropic" | "openai" | "groq"

CONFIGS = {
    "ollama": {
        "base_url": "http://localhost:11434/v1",
        "api_key":  "ollama",
        "model":    "qwen2.5:7b",
    },
    "anthropic": {
        "base_url": "https://api.anthropic.com/v1",
        "api_key":  os.getenv("ANTHROPIC_API_KEY", ""),
        "model":    "claude-3-5-haiku-20241022",
    },
    "openai": {
        "base_url": None,
        "api_key":  os.getenv("OPENAI_API_KEY", ""),
        "model":    "gpt-4o-mini",
    },
    "groq": {
        "base_url": "https://api.groq.com/openai/v1",
        "api_key":  os.getenv("GROQ_API_KEY", ""),
        "model":    "llama-3.3-70b-versatile",
    },
    "nvidia": {
        "base_url": "https://integrate.api.nvidia.com/v1",
        "api_key":  os.getenv("NVIDIA_API_KEY", ""),
        "model":    "meta/llama-3.3-70b-instruct",  # o cualquier otro del catálogo
    },
}

cfg = CONFIGS[BACKEND]
llm_kwargs = {"model": cfg["model"], "api_key": cfg["api_key"]}
if cfg["base_url"]:
    llm_kwargs["base_url"] = cfg["base_url"]

llm = ChatOpenAI(**llm_kwargs)
print(f"✅ Backend: {BACKEND.upper()} | Modelo: {cfg['model']}")

✅ Backend: NVIDIA | Modelo: meta/llama-3.3-70b-instruct


---
## Sección 3: Herramienta Compartida y Helper

Definimos `web_search` y `build_react_agent` — el factory que todos los patrones usarán.

In [7]:
@tool
def web_search(query: str) -> str:
    """Busca información actualizada en la web."""
    try:
        results = DDGS().text(query, max_results=3)
        if not results:
            return "No se encontraron resultados."
        output = []
        for r in results:
            output.append(f"**{r['title']}**\n{r['body']}")
        return "\n\n".join(output)
    except Exception as e:
        return f"Error al buscar: {e}"



search_tool_node = ToolNode([web_search])
llm_with_search  = llm.bind_tools([web_search])


def build_react_agent(system_prompt: str, tools: list):
    """Factory: construye un mini grafo ReAct con un system prompt dado."""
    _llm = llm.bind_tools(tools)
    _tool_node = ToolNode(tools)

    def _agent(state: MessagesState):
        msgs = state["messages"]
        if not any(isinstance(m, SystemMessage) for m in msgs):
            msgs = [SystemMessage(content=system_prompt)] + msgs
        return {"messages": [_llm.invoke(msgs)]}

    def _router(state: MessagesState) -> Literal["tools", "__end__"]:
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return "__end__"

    g = StateGraph(MessagesState)
    g.add_node("agent", _agent)
    g.add_node("tools", _tool_node)
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", _router)
    g.add_edge("tools", "agent")
    return g.compile()


def invoke_llm(system: str, user: str) -> str:
    """Helper: llama al LLM sin tools y devuelve el texto."""
    msgs = [SystemMessage(content=system), HumanMessage(content=user)]
    return llm.invoke(msgs).content


print("Herramienta y helpers listos.")

Herramienta y helpers listos.


---
## Patrón 1: Orquestador LLM

El modelo coordinador decide dinámicamente qué sub-agente invocar.
Cada sub-agente es un grafo ReAct independiente envuelto en un `@tool`.

```
Usuario
  └─► Coordinador (LLM)
         ├─► research_agent   (tool call → grafo ReAct interno)
         └─► summarizer_agent (tool call → grafo ReAct interno)
```

El coordinador llama a `research_agent` exactamente igual
que llamaría a `web_search`. La diferencia es que `research_agent` internamente
ejecuta otro grafo completo.

In [8]:
_research_graph   = build_react_agent(
    "Eres un agente de investigación. Usa web_search para encontrar "
    "2-3 datos relevantes. Presenta hallazgos con fuentes.", [web_search]
)
_summarizer_graph = build_react_agent(
    "Eres un agente de síntesis. Crea un resumen en 3-5 puntos clave "
    "con viñetas. No uses herramientas.", []
)


@tool
def research_agent(topic: str) -> str:
    """Investiga un tema en la web y devuelve hallazgos con fuentes."""
    result = _research_graph.invoke(
        {"messages": [HumanMessage(content=f"Investiga: {topic}")]}
    )
    return result["messages"][-1].content


@tool
def summarizer_agent(research_findings: str) -> str:
    """Resume hallazgos de investigación en puntos clave con viñetas."""
    result = _summarizer_graph.invoke(
        {"messages": [HumanMessage(content=f"Resume:\n{research_findings}")]}
    )
    return result["messages"][-1].content


orchestrator_tools = [research_agent, summarizer_agent]
orchestrator_graph = build_react_agent(
    "Eres un coordinador de investigación. Para responder:\n"
    "1. Llama a `research_agent` para obtener información actualizada.\n"
    "2. Llama a `summarizer_agent` con los hallazgos.\n"
    "3. Presenta el resumen final al usuario.",
    orchestrator_tools,
)


def run_orchestrator(query: str) -> str:
    result = orchestrator_graph.invoke(
        {"messages": [HumanMessage(content=query)]}
    )
    return result["messages"][-1].content


# Demo
print("=" * 60)
print("PATRÓN 1: Orquestador LLM")
print("=" * 60)
orchestrator_result = run_orchestrator(
    "¿Cuáles son los últimos avances en computación cuántica?"
)
print(orchestrator_result[:400], "..." if len(orchestrator_result) > 400 else "")

PATRÓN 1: Orquestador LLM


Los últimos avances en computación cuántica incluyen:

*   Desarrollos en la tecnología de bits cuánticos (cúbits) para procesar información de manera más eficiente que las computadoras tradicionales.
*   Aplicaciones en criptografía, optimización y simulación de sistemas complejos.
*   Investigaciones en la simulación de fenómenos subatómicos y su potencial para revolucionar diversos campos.

Fue ...


---
## Patrón 2: Pipeline Secuencial

Nodos encadenados con orden garantizado. El estado tipado (`BlogState`)
lleva el output de cada paso al siguiente.

```
topic → [outline_node] → outline
                  └──► [writer_node] → draft
                               └──► [editor_node] → final
```

**El estado es un `TypedDict` — cada nodo lee y escribe
campos específicos. LangGraph garantiza la secuencia sin que el LLM decida el orden.

In [9]:
class BlogState(TypedDict):
    topic:   str
    outline: str
    draft:   str
    final:   str


def outline_node(state: BlogState) -> dict:
    print("  [OutlineAgent] generando outline...")
    result = invoke_llm(
        "Crea un outline de blog con título, hook, 3-5 secciones y conclusión.",
        f"Tema: {state['topic']}"
    )
    return {"outline": result}


def writer_node(state: BlogState) -> dict:
    print("  [WriterAgent] escribiendo borrador...")
    result = invoke_llm(
        "Escribe un blog de 200-300 palabras siguiendo el outline estrictamente.",
        f"Outline:\n{state['outline']}"
    )
    return {"draft": result}


def editor_node(state: BlogState) -> dict:
    print("  [EditorAgent] editando...")
    result = invoke_llm(
        "Corrige errores, mejora flujo y claridad. Mantén la longitud. Solo el texto.",
        f"Borrador:\n{state['draft']}"
    )
    return {"final": result}


seq_builder = StateGraph(BlogState)
seq_builder.add_node("outline", outline_node)
seq_builder.add_node("writer",  writer_node)
seq_builder.add_node("editor",  editor_node)
seq_builder.add_edge(START,     "outline")
seq_builder.add_edge("outline", "writer")
seq_builder.add_edge("writer",  "editor")
seq_builder.add_edge("editor",  END)
seq_graph = seq_builder.compile()


def run_sequential(topic: str) -> BlogState:
    return seq_graph.invoke({"topic": topic, "outline": "", "draft": "", "final": ""})


# Demo
print("=" * 60)
print("PATRÓN 2: Pipeline Secuencial")
print("=" * 60)
seq_result = run_sequential(
    "Los beneficios de los sistemas multi-agente para desarrolladores de software"
)
print("\nBlog final (primeros 300 chars):")
print(seq_result["final"][:300], "...")

PATRÓN 2: Pipeline Secuencial
  [OutlineAgent] generando outline...


  [WriterAgent] escribiendo borrador...
  [EditorAgent] editando...

Blog final (primeros 300 chars):
**Desbloquea el Poder de los Sistemas Multi-Agente: Beneficios para Desarrolladores de Software**

¿Alguna vez has imaginado un futuro donde los sistemas de software puedan trabajar juntos de manera autónoma y eficiente para resolver problemas complejos? Los sistemas multi-agente están revolucionand ...


---
## Patrón 3: Ejecución Paralela con `Send()`

LangGraph ejecuta ramas independientes en paralelo usando `Send()`.
Un edge de fan-out lanza un nodo por cada dominio simultáneamente.

```
Input
  ├─► [TechResearcher]    ─┐
  ├─► [HealthResearcher]  ─┼─► [Aggregator] → Resumen
  └─► [FinanceResearcher] ─┘
       (los 3 corren en paralelo)
```

**`Annotated[list, operator.add]` en el estado hace que
los reportes de cada rama se acumulen automáticamente en la lista — no hay
que coordinar manualmente los resultados.

In [10]:
from langgraph.types import Send


class ParallelState(TypedDict):
    domains:  list[str]
    reports:  Annotated[list, operator.add]   # acumulación automática
    summary:  str


class DomainState(TypedDict):
    domain:       str
    instructions: str


DOMAIN_INSTRUCTIONS = {
    "tech":    "Investiga tendencias IA/ML. 3 desarrollos, empresas, impacto. Máx 100 palabras.",
    "health":  "Investiga avances médicos recientes. 3 avances, aplicaciones, plazos. Máx 100 palabras.",
    "finance": "Investiga tendencias fintech. 3 tendencias, mercado, perspectivas. Máx 100 palabras.",
}


def research_domain_node(state: DomainState) -> dict:
    print(f"  [{state['domain'].upper()}Researcher] investigando...")
    result = _research_graph.invoke(
        {"messages": [HumanMessage(content=state["instructions"])]}
    )
    report = f"**{state['domain'].capitalize()}**\n{result['messages'][-1].content}"
    return {"reports": [report]}


def aggregate_node(state: ParallelState) -> dict:
    print("  [Aggregator] sintetizando...")
    combined = "\n\n".join(state["reports"])
    summary = invoke_llm(
        "Combina los reportes en un resumen ejecutivo de ~200 palabras. "
        "Destaca temas comunes y los 3 takeaways más importantes.",
        combined
    )
    return {"summary": summary}


def fan_out(state: ParallelState) -> list[Send]:
    return [
        Send("research_domain", {
            "domain":       domain,
            "instructions": DOMAIN_INSTRUCTIONS[domain],
        })
        for domain in state["domains"]
    ]


par_builder = StateGraph(ParallelState)
par_builder.add_node("research_domain", research_domain_node)
par_builder.add_node("aggregate",       aggregate_node)
par_builder.add_conditional_edges(START, fan_out, ["research_domain"])
par_builder.add_edge("research_domain", "aggregate")
par_builder.add_edge("aggregate", END)
par_graph = par_builder.compile()


def run_parallel(domains: list[str] = None) -> ParallelState:
    domains = domains or ["tech", "health", "finance"]
    t0 = time.time()
    result = par_graph.invoke({"domains": domains, "reports": [], "summary": ""})
    print(f"  ⏱️  Completado en {time.time()-t0:.1f}s")
    return result


# Demo
print("=" * 60)
print("PATRÓN 3: Ejecución Paralela")
print("=" * 60)
par_result = run_parallel()
print("\nResumen ejecutivo (primeros 300 chars):")
print(par_result["summary"][:300], "...")

PATRÓN 3: Ejecución Paralela
  [TECHResearcher] investigando...
  [HEALTHResearcher] investigando...
  [FINANCEResearcher] investigando...
  [Aggregator] sintetizando...
  ⏱️  Completado en 27.8s

Resumen ejecutivo (primeros 300 chars):
**Resumen Ejecutivo**

En este resumen, se destacan las tendencias y avances en tres áreas clave: tecnología, salud y finanzas. En tecnología, la inteligencia artificial (IA) y el aprendizaje automático (ML) están transformando la forma en que las empresas operan, con un enfoque en la automatización ...


---
## Patrón 4: Loop de Refinamiento

Grafo con ciclo: el `critic_node` evalúa y la arista condicional `loop_router`
decide si continuar o terminar. La condición de salida está en Python, no en el LLM.

```
prompt → [initial_writer] → historia v1
                 ↓
     ┌──── CICLO ──────────────────┐
     │  [critic] → feedback        │
     │      ↓                      │
     │  approved o max_iter?        │
     │    Sí → END                  │
     │    No → [refiner] → v+1     │
     └─────────────────────────────┘
```

**Clave pedagógica:** `loop_router` retorna `END` o `"refiner"` — es una función
Python normal, no depende del LLM para decidir si el loop continúa.

In [11]:
class StoryState(TypedDict):
    prompt:     str
    story:      str
    critique:   str
    iteration:  int
    approved:   bool
    max_iter:   int


def initial_writer_node(state: StoryState) -> dict:
    print("  [InitialWriter] escribiendo primera versión...")
    story = invoke_llm(
        "Escribe el primer borrador de una historia corta (100-150 palabras). Solo el texto.",
        state["prompt"]
    )
    return {"story": story, "iteration": 0}


def critic_node(state: StoryState) -> dict:
    i = state["iteration"] + 1
    print(f"  [Critic] evaluando (iteración {i}/{state['max_iter']})...")
    critique = invoke_llm(
        "Eres un crítico constructivo. Evalúa trama, personajes y ritmo.\n"
        "- Si es sólida y completa, responde EXACTAMENTE: APPROVED\n"
        "- Si no, da 2-3 sugerencias específicas y accionables.",
        f"Historia:\n{state['story']}"
    )
    approved = "APPROVED" in critique.upper()
    if approved:
        print(f"  ✅ Historia aprobada en iteración {i}.")
    return {"critique": critique, "iteration": i, "approved": approved}


def refiner_node(state: StoryState) -> dict:
    print("  [Refiner] reescribiendo con feedback...")
    story = invoke_llm(
        "Reescribe la historia incorporando TODO el feedback del crítico. Solo el texto.",
        f"Historia:\n{state['story']}\n\nCrítica:\n{state['critique']}"
    )
    return {"story": story}


def loop_router(state: StoryState) -> Literal["refiner", END]:
    if state["approved"] or state["iteration"] >= state["max_iter"]:
        return END
    return "refiner"


loop_builder = StateGraph(StoryState)
loop_builder.add_node("initial_writer", initial_writer_node)
loop_builder.add_node("critic",         critic_node)
loop_builder.add_node("refiner",        refiner_node)
loop_builder.add_edge(START,            "initial_writer")
loop_builder.add_edge("initial_writer", "critic")
loop_builder.add_conditional_edges("critic", loop_router)
loop_builder.add_edge("refiner",        "critic")
loop_graph = loop_builder.compile()


def run_refinement_loop(prompt: str, max_iter: int = 3) -> StoryState:
    return loop_graph.invoke({
        "prompt":    prompt,
        "story":     "",
        "critique":  "",
        "iteration": 0,
        "approved":  False,
        "max_iter":  max_iter,
    })


# Demo
print("=" * 60)
print("PATRÓN 4: Loop de Refinamiento (Escritor ⇆ Crítico)")
print("=" * 60)
loop_result = run_refinement_loop(
    "Un guardián de faro que descubre un mapa que brilla en la oscuridad.",
    max_iter=3,
)
print(f"\nHistoria final ({loop_result['iteration']} iteraciones | "
      f"aprobada: {loop_result['approved']}):")
print(loop_result["story"][:400], "...")

PATRÓN 4: Loop de Refinamiento (Escritor ⇆ Crítico)
  [InitialWriter] escribiendo primera versión...
  [Critic] evaluando (iteración 1/3)...
  [Refiner] reescribiendo con feedback...
  [Critic] evaluando (iteración 2/3)...
  [Refiner] reescribiendo con feedback...
  [Critic] evaluando (iteración 3/3)...

Historia final (3 iteraciones | aprobada: False):
La noche era oscura y el viento azotaba con fuerza contra el faro, un lugar que Jack, el guardián, había llamado hogar durante los últimos cinco años. Jack, un hombre de cincuenta años con una barba gris y ojos cansados, había sido un marinero durante gran parte de su vida, pero después de un accidente que le había dejado una cojera permanente, se había visto obligado a buscar un trabajo en tierra ...


---
## 📊 Sección 5: Evaluación de Agentes

Evaluar agentes es diferente a evaluar modelos clásicos de ML:
no hay un ground truth numérico fijo. Implementamos tres estrategias:

| Estrategia | Descripción | Cuándo usarla |
|---|---|---|
| **Métricas de proceso** | tool precision/recall, latencia | Siempre — sin costo extra |
| **LLM-as-judge** | otro LLM puntúa 1-5 con justificación | Calidad semántica |
| **Reference-based** | comparar contra respuesta esperada | Cuando tienes ground truth |

In [ ]:
import json
import re
from dataclasses import dataclass, field


# ── 5.1 Casos de prueba ───────────────────────────────────────────────────────

@dataclass
class TestCase:
    id:               str
    query:            str
    expected_tools:   list[str]
    reference_answer: str
    min_words:        int = 20


TEST_CASES = [
    TestCase(
        id="tc_no_tool",
        query="¿Cuántas lunas tiene Marte?",
        expected_tools=[],
        reference_answer="Marte tiene dos lunas: Fobos y Deimos.",
        min_words=5,
    ),
    TestCase(
        id="tc_weather",
        query="¿Cómo está el clima en Medellín?",
        expected_tools=["get_weather"],
        reference_answer="temperatura actual en Medellín",
        min_words=10,
    ),
    TestCase(
        id="tc_calculate",
        query="¿Cuánto es el 18% de 3450000?",
        expected_tools=["calculate"],
        reference_answer="621000",
        min_words=5,
    ),
    TestCase(
        id="tc_search",
        query="¿Qué es LangGraph?",
        expected_tools=["web_search"],
        reference_answer="framework grafo estado agentes",
        min_words=30,
    ),
    TestCase(
        id="tc_multi_tool",
        query="¿Cuál es el precio de Bitcoin y cuánto costarían 2.5 BTC?",
        expected_tools=["web_search", "calculate"],
        reference_answer="precio Bitcoin multiplicado por 2.5",
        min_words=20,
    ),
]

print(f"✅ {len(TEST_CASES)} casos de prueba definidos.")

### 5.2 Métricas de proceso y agente de evaluación

Definimos `EvalResult`, `compute_tool_metrics` y el grafo de evaluación
que captura qué tools usó el agente.

In [ ]:
@dataclass
class EvalResult:
    test_id:        str
    query:          str
    response:       str
    tools_used:     list[str]
    expected_tools: list[str]
    tool_precision: float
    tool_recall:    float
    tool_f1:        float
    word_count:     int
    latency_s:      float
    llm_score:      float = 0.0
    llm_rationale:  str   = ""


def compute_tool_metrics(used: list[str], expected: list[str]) -> tuple[float, float, float]:
    used_set     = set(used)
    expected_set = set(expected)
    tp = len(used_set & expected_set)
    fp = len(used_set - expected_set)
    fn = len(expected_set - used_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else (1.0 if not expected_set else 0.0)
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1


# Agente de evaluación con tools spy (capturan qué se llamó)
from langchain_core.tools import tool as lc_tool

@lc_tool
def web_search_eval(query: str) -> str:
    """Busca información en la web."""
    return web_search.invoke({"query": query})

@lc_tool
def get_weather_eval(city: str) -> str:
    """Obtiene el clima actual de una ciudad."""
    try:
        r = httpx.get(f"https://wttr.in/{urllib.parse.quote(city)}?format=3", timeout=8)
        return r.text.strip()
    except Exception as e:
        return f"Error: {e}"

@lc_tool
def calculate_eval(expression: str) -> str:
    """Evalúa una expresión matemática."""
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return "Error: solo expresiones numéricas básicas."
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

EVAL_TOOLS = [web_search_eval, get_weather_eval, calculate_eval]
eval_graph  = build_react_agent(
    "Eres un asistente útil. Usa las herramientas cuando sea necesario.",
    EVAL_TOOLS
)


def run_agent_for_eval(query: str) -> tuple[str, list[str], float]:
    t0     = time.time()
    result = eval_graph.invoke({"messages": [HumanMessage(content=query)]})
    lat    = time.time() - t0
    tools_used = []
    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                name = tc["name"].replace("_eval", "")
                tools_used.append(name)
    return result["messages"][-1].content, tools_used, lat


print("✅ Métricas y agente de evaluación listos.")

### 5.3 LLM-as-judge

Un segundo LLM puntúa la calidad de la respuesta en escala 1-5 con justificación.

In [ ]:
JUDGE_PROMPT = """\
Eres un evaluador experto. Puntúa la siguiente respuesta de un agente de IA.

Pregunta del usuario: {query}
Respuesta del agente: {response}
Respuesta de referencia (orientativa): {reference}

Criterios:
- Precisión factual
- Completitud
- Claridad
- Uso apropiado de herramientas

Responde ÚNICAMENTE con JSON:
{{"score": <1-5>, "rationale": "<1-2 oraciones>"}}
"""


def llm_judge(query: str, response: str, reference: str) -> tuple[float, str]:
    prompt = JUDGE_PROMPT.format(query=query, response=response, reference=reference)
    try:
        raw   = llm.invoke([HumanMessage(content=prompt)]).content
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            data = json.loads(match.group())
            return float(data.get("score", 0)), str(data.get("rationale", ""))
    except Exception as e:
        print(f"    ⚠️  LLM judge error: {e}")
    return 0.0, "Error al obtener puntuación"


print("✅ LLM-as-judge listo.")

### 5.4 Ejecutar evaluación y reporte

In [ ]:
def run_evaluation(test_cases: list[TestCase], use_llm_judge: bool = True) -> list[EvalResult]:
    results = []
    for tc in test_cases:
        print(f"\n  Evaluando [{tc.id}]: {tc.query[:60]}...")
        response, tools_used, latency = run_agent_for_eval(tc.query)
        precision, recall, f1 = compute_tool_metrics(tools_used, tc.expected_tools)
        result = EvalResult(
            test_id=tc.id, query=tc.query, response=response,
            tools_used=tools_used, expected_tools=tc.expected_tools,
            tool_precision=precision, tool_recall=recall, tool_f1=f1,
            word_count=len(response.split()), latency_s=latency,
        )
        if use_llm_judge:
            result.llm_score, result.llm_rationale = llm_judge(
                tc.query, response, tc.reference_answer
            )
        results.append(result)
        print(f"    tools={tools_used} | P={precision:.2f} R={recall:.2f} "
              f"| score={result.llm_score}/5 | lat={latency:.1f}s")
    return results


def print_eval_report(results: list[EvalResult]) -> None:
    print("\n" + "=" * 80)
    print("REPORTE DE EVALUACIÓN")
    print("=" * 80)
    print(f"{'ID':<18} {'P':>5} {'R':>5} {'F1':>5} {'Score':>6} {'Words':>6} {'Lat(s)':>7}")
    print("-" * 80)
    for r in results:
        print(f"{r.test_id:<18} {r.tool_precision:>5.2f} {r.tool_recall:>5.2f} "
              f"{r.tool_f1:>5.2f} {r.llm_score:>6.1f} {r.word_count:>6} {r.latency_s:>7.1f}")
    print("-" * 80)
    n = len(results)
    print(f"{'PROMEDIO':<18} "
          f"{sum(r.tool_precision for r in results)/n:>5.2f} "
          f"{sum(r.tool_recall for r in results)/n:>5.2f} "
          f"{sum(r.tool_f1 for r in results)/n:>5.2f} "
          f"{sum(r.llm_score for r in results)/n:>6.1f} "
          f"{'':>6} {sum(r.latency_s for r in results)/n:>7.1f}")
    print("=" * 80)
    print("P=Precision | R=Recall | F1 | Score=LLM-judge(1-5) | Words | Lat=latencia")
    print("\n── Justificaciones LLM-judge ──────────────────────────────────")
    for r in results:
        if r.llm_rationale:
            print(f"\n[{r.test_id}] score={r.llm_score}")
            print(f"  Q: {r.query[:70]}")
            print(f"  → {r.llm_rationale}")
    failures = [r for r in results if r.tool_f1 < 1.0]
    if failures:
        print("\n── Casos con herramientas incorrectas ──────────────────────")
        for r in failures:
            print(f"  [{r.test_id}] esperadas={r.expected_tools} usadas={r.tools_used} F1={r.tool_f1:.2f}")


# Ejecutar
# use_llm_judge=False si solo quieres métricas de proceso (más rápido)
print("=" * 60)
print("SECCIÓN 5: Evaluación del Agente ReAct")
print("=" * 60)
eval_results = run_evaluation(TEST_CASES, use_llm_judge=True)
print_eval_report(eval_results)

---
## ✅ Resumen

Implementaste los cuatro patrones de arquitectura multi-agente con LangGraph.

| Patrón | Nodos LangGraph | Cuándo usarlo |
|---|---|---|
| Orquestador LLM | `agent` + `ToolNode` con sub-agentes | Flujo dinámico |
| Sequential | nodos en cadena + `TypedDict` state | Pasos que se construyen |
| Parallel | fan-out con `Send()` | Tareas independientes |
| Loop | ciclo + edge condicional | Refinamiento iterativo |

Los frameworks implementan exactamente estos mismos patrones.
Ahora que entiendes el núcleo, cualquier framework te resultará transparente.